# GBM tuning
Decision metric = `sb_cv.cv_ts_auc` **mean**. Search uses a *fast* CV config (3 folds, 1 seed, ~2 min/run); finalists re-checked on **full CV** (5 folds, 3 seeds) and the **fixed holdout + reduced test** (comparable to memory's 0.5793 / 0.5764).

**Gate:** keep a change only if CV-mean gain > CV-std (~0.01). We're near the causal last-step ceiling (~0.607 full-res ≈ 0.579 subsample), so expect small moves.

Run with the `structural-break` conda env, cwd = `experiments/`.

In [ ]:
import numpy as np, pandas as pd, lightgbm as lgb, time
import sb_common as C, sb_cv as cv
from sb_model import PARAMS, rank01, eq_series_weight, true_step

X, idx, cols = cv.load_bank()   # cached subsample bank, all train series
y = cv.load_y()
print('bank', X.shape)

FAST = dict(seeds=(0,),     n_splits=3)   # cheap search config
FULL = dict(seeds=(0,1,2),  n_splits=5)   # finalist config

def score(params, tag='', **kw):
    t0 = time.time()
    m, s, f = cv.cv_ts_auc(X, idx, y, params=params, **kw)
    print('%-14s cv %.4f +/- %.4f  %.0fs  folds=%s' % (
        tag, m, s, time.time()-t0, [round(x,4) for x in f]))
    return m, s

## 1. Baseline (current PARAMS)

In [ ]:
base_m, base_s = score(PARAMS, 'baseline', **FAST)

## 2. Slow-lr / more-trees (winner ~+0.011 recipe)

In [ ]:
SLOW = {**PARAMS, 'learning_rate': 0.01, 'n_estimators': 2500,
        'reg_alpha': 1.0, 'reg_lambda': 3.0}
score(SLOW, 'slow-lr', **FAST)

## 3. Optuna search (fast-CV objective)

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(t):
    p = {**PARAMS,
         'learning_rate':     t.suggest_float('lr', 0.005, 0.06, log=True),
         'n_estimators':      t.suggest_int('n_est', 400, 3000),
         'num_leaves':        t.suggest_int('leaves', 31, 127),
         'min_child_samples': t.suggest_int('min_child', 100, 600),
         'colsample_bytree':  t.suggest_float('colsample', 0.5, 1.0),
         'reg_lambda':        t.suggest_float('l2', 0.0, 5.0)}
    m, _ = cv.cv_ts_auc(X, idx, y, params=p, **FAST)
    return m

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=True)
print('best fast-cv %.4f' % study.best_value)
print(study.best_params)

## 4. Bag-size sweep on the best params (full CV)

In [ ]:
bp = study.best_params
best = {**PARAMS, 'learning_rate': bp['lr'], 'n_estimators': bp['n_est'],
        'num_leaves': bp['leaves'], 'min_child_samples': bp['min_child'],
        'colsample_bytree': bp['colsample'], 'reg_lambda': bp['l2']}

score(PARAMS, 'base FULL', **FULL)                      # anchor at full config
for nb in (3, 5, 8):
    score(best, 'best bag%d' % nb, seeds=tuple(range(nb)), n_splits=5)

## 5. Final validation — fixed holdout + reduced test
Trains on the fixed `tr_ids` split (5-seed bag) and scores the same two eval sets memory tracks. Keep `best` only if it beats **0.5793** (holdout) without hurting **0.5764** (reduced).

In [ ]:
tr_ids = np.load('tr_ids.npy')
ids = idx.get_level_values('id').to_numpy()
mtr = np.isin(ids, tr_ids)
yv  = y.reindex(idx).to_numpy().astype(np.int8)
wtr = eq_series_weight(idx[mtr])

def load_eval(path):
    F = pd.read_parquet(path)[cols]
    return (F.to_numpy(np.float32),
            y.reindex(F.index).to_numpy().astype(np.int8),
            true_step(F.index, y))

Xh, yh, th = load_eval('feat_final_holdout.parquet')
Xr, yr, tr_ = load_eval('feat_final_reduced.parquet')

def final_eval(params, tag, nseed=5):
    ph, pr = [], []
    for s in range(nseed):
        m = lgb.LGBMClassifier(random_state=s, **params).fit(X[mtr], yv[mtr], sample_weight=wtr)
        ph.append(m.predict_proba(Xh)[:,1]); pr.append(m.predict_proba(Xr)[:,1])
    h = C.ts_auc_arrays(np.mean([rank01(q) for q in ph],0), yh, th)
    r = C.ts_auc_arrays(np.mean([rank01(q) for q in pr],0), yr, tr_)
    print('%-9s holdout %.4f (base 0.5793) | reduced %.4f (base 0.5764)' % (tag, h, r))
    return h, r

final_eval(PARAMS, 'baseline')
final_eval(best,   'tuned')

## 6. Persist if it wins
If `tuned` clears the gate, append to `sb_model.py`:
```python
PARAMS_TUNED = { ... best ... }
```

In [ ]:
# print a paste-ready dict once the gate is cleared
print('PARAMS_TUNED = ' + repr(best))